# Map eligible stocks to ICB sectors

Add Industry, Supersector, Sector and Subsector from `Data/stock_sector_mapping.csv`. The saved ICB snapshot is applied to the full price history.

Match by ISIN first. For an ISIN without a classification, use a shared `CompanyId` or `Internal code` when the mapped ISINs agree on all four labels.

In [1]:
import csv
from pathlib import Path
from tempfile import NamedTemporaryFile

import pandas as pd

data_dir = Path("Data") if Path("Data").is_dir() else Path("../Data")
eligible_dir = data_dir / "eligible"
source = eligible_dir / "BMG9156K1018_10.09.2026_14.59.34_trading_days_eligible.csv"
output = eligible_dir / "data_mapped.csv"
levels = [f"icb_{level}_name" for level in ("industry", "supersector", "sector", "subsector")]

identifiers = ["CompanyId", "Internal code"]
eligible = pd.read_csv(source, sep=";", usecols=["ISIN", *identifiers], dtype="string")
snapshot = pd.read_csv(data_dir / "stock_sector_mapping.csv", usecols=["ISIN", *levels], dtype="string")
by_isin = snapshot.set_index("ISIN", verify_integrity=True).dropna(subset=levels)

keys = ["Identifier type", "Identifier"]
isin_identifiers = eligible.melt(
    id_vars="ISIN", value_vars=identifiers, var_name=keys[0], value_name=keys[1]
).dropna().drop_duplicates()
mapped_identifiers = isin_identifiers.join(by_isin, on="ISIN").dropna(subset=levels)
labels_by_identifier = mapped_identifiers[[*keys, *levels]].drop_duplicates()
inherited = isin_identifiers.merge(labels_by_identifier, on=keys)[["ISIN", *levels]].drop_duplicates()
inherited = inherited.drop_duplicates("ISIN", keep=False).set_index("ISIN")

eligible_isins = pd.Index(eligible["ISIN"].unique(), name="ISIN")
exact_matches = by_isin.reindex(eligible_isins)
classifications = exact_matches.combine_first(inherited.reindex(eligible_isins))[levels]
exact_count = exact_matches[levels[0]].count()
fallback_count = classifications[levels[0]].count() - exact_count
unmapped = classifications.index[classifications[levels[0]].isna()]

print(f"ISINs: {exact_count} exact matches, {fallback_count} linked matches, {len(unmapped)} unmapped.")
if len(unmapped):
    print("Unmapped ISINs:", ", ".join(unmapped))

ISINs: 127 exact matches, 13 linked matches, 0 unmapped.


## Export `data_mapped.csv`

Append the four ICB labels to each source row. Keep the original `Sector` column and leave unresolved labels blank.

In [2]:
labels_by_isin = {
    isin: labels for isin, *labels in classifications.fillna("").itertuples(name=None)
}

with NamedTemporaryFile(
    mode="w", encoding="utf-8", newline="", dir=eligible_dir, suffix=".tmp", delete=False
) as dst:
    temporary = Path(dst.name)
    try:
        with source.open(encoding="utf-8", newline="") as src:
            reader = csv.reader(src, delimiter=";")
            writer = csv.writer(dst, delimiter=";", lineterminator="\n")
            header = next(reader)
            isin_index = header.index("ISIN")
            writer.writerow([*header, *levels])
            for row in reader:
                writer.writerow([*row, *labels_by_isin[row[isin_index]]])
        dst.close()
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)

print(f"Wrote {len(eligible):,} rows to {output}.")

Wrote 298,839 rows to ../Data/eligible/data_mapped.csv.
